# Data Cleaning

En este notebook se realiza la exploración y limpieza inicial de los datos.

El objetivo es preparar la información para su posterior uso en modelos de Machine Learning.

In [16]:
# Importar librerías

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when
from pyspark.ml.feature import Imputer

## Crear la sesión de Apache Spark

Se crea una sesión de Spark para el procesamiento distribuido de datos.

In [17]:
# Crear sesión de Spark

spark = (
    SparkSession.builder
    .appName("FinancialDigitalTwin")
    .getOrCreate()
)

## Cargar los conjuntos de datos

In [18]:
# Cargar datasets

cards_spark = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../data/modified/cards_data.csv")
)

transactions_spark = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../data/modified/transactions_data.csv")
)

users_spark = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../data/modified/users_data.csv")
)

In [19]:
# Mostrar primeras filas

cards_spark.show(5)

+----+---------+----------+---------------+-----------+-------+---+--------+----------------+------------+--------------+---------------------+----------------+
|  id|client_id|card_brand|      card_type|card_number|expires|cvv|has_chip|num_cards_issued|credit_limit|acct_open_date|year_pin_last_changed|card_on_dark_web|
+----+---------+----------+---------------+-----------+-------+---+--------+----------------+------------+--------------+---------------------+----------------+
|4524|      825|      Visa|          Debit| 4.34468E15| dic-22|623|     YES|               2|     $24,295|        sep-02|                 2008|              No|
|2731|      825|      Visa|          Debit| 4.95697E15| dic-20|393|     YES|               2|     $21,968|        abr-14|                 2014|              No|
|3701|      825|      Visa|          Debit| 4.58231E15| feb-24|719|     YES|               2|     $46,414|        jul-03|                 2004|              No|
|  42|      825|      Visa|       

In [20]:
transactions_spark.show(5)

+-------+----------------+---------+-------+-------+-----------------+-----------+-------------+--------------+-----+----+------+
|     id|            date|client_id|card_id| amount|         use_chip|merchant_id|merchant_city|merchant_state|  zip| mcc|errors|
+-------+----------------+---------+-------+-------+-----------------+-----------+-------------+--------------+-----+----+------+
|7475327|01/01/2010 00:01|     1556|   2972|-$77.00|Swipe Transaction|      59935|       Beulah|            ND|58523|5499|  NULL|
|7475328|01/01/2010 00:02|      561|   4575| $14.57|Swipe Transaction|      67570|   Bettendorf|            IA|52722|5311|  NULL|
|7475329|01/01/2010 00:02|     1129|    102| $80.00|Swipe Transaction|      27092|        Vista|            CA|92084|4829|  NULL|
|7475331|01/01/2010 00:05|      430|   2860|$200.00|Swipe Transaction|      27092|  Crown Point|            IN|46307|4829|  NULL|
|7475332|01/01/2010 00:06|      848|   3915| $46.41|Swipe Transaction|      13051|      Ha

In [21]:
users_spark.show(5)

+----+-----------+--------------+----------+-----------+------+--------------------+--------+---------+-----------------+-------------+----------+------------+----------------+
|  id|current_age|retirement_age|birth_year|birth_month|gender|             address|latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|
+----+-----------+--------------+----------+-----------+------+--------------------+--------+---------+-----------------+-------------+----------+------------+----------------+
| 825|         53|            66|      1966|         11|Female|       462 Rose Lane|   34.15|  -117.76|          $29,278|      $59,696|  $127,613|         787|               5|
|1746|         53|            68|      1966|         12|Female|3606 Federal Boul...|   40.76|   -73.74|          $37,891|      $77,254|  $191,349|         701|               5|
|1718|         81|            67|      1938|         11|Female|     766 Third Drive|   34.02|  -117.89|          $2

## Explorar la información

In [22]:
cards_spark.printSchema()

root
 |-- id: integer (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_number: double (nullable = true)
 |-- expires: string (nullable = true)
 |-- cvv: integer (nullable = true)
 |-- has_chip: string (nullable = true)
 |-- num_cards_issued: integer (nullable = true)
 |-- credit_limit: string (nullable = true)
 |-- acct_open_date: string (nullable = true)
 |-- year_pin_last_changed: integer (nullable = true)
 |-- card_on_dark_web: string (nullable = true)



In [23]:
transactions_spark.printSchema()

root
 |-- id: integer (nullable = true)
 |-- date: string (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- card_id: integer (nullable = true)
 |-- amount: string (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- mcc: integer (nullable = true)
 |-- errors: string (nullable = true)



In [24]:
users_spark.printSchema()

root
 |-- id: integer (nullable = true)
 |-- current_age: integer (nullable = true)
 |-- retirement_age: integer (nullable = true)
 |-- birth_year: integer (nullable = true)
 |-- birth_month: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- address: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- per_capita_income: string (nullable = true)
 |-- yearly_income: string (nullable = true)
 |-- total_debt: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- num_credit_cards: integer (nullable = true)



## Identificar valores faltantes

In [25]:
cards_spark.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in cards_spark.columns
]).show()

+---+---------+----------+---------+-----------+-------+---+--------+----------------+------------+--------------+---------------------+----------------+
| id|client_id|card_brand|card_type|card_number|expires|cvv|has_chip|num_cards_issued|credit_limit|acct_open_date|year_pin_last_changed|card_on_dark_web|
+---+---------+----------+---------+-----------+-------+---+--------+----------------+------------+--------------+---------------------+----------------+
|  1|        1|        72|       57|         63|     66| 68|      68|              62|          91|            55|                   72|              76|
+---+---------+----------+---------+-----------+-------+---+--------+----------------+------------+--------------+---------------------+----------------+



In [26]:
transactions_spark.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in transactions_spark.columns
]).show()

+---+----+---------+-------+------+--------+-----------+-------------+--------------+----+---+------+
| id|date|client_id|card_id|amount|use_chip|merchant_id|merchant_city|merchant_state| zip|mcc|errors|
+---+----+---------+-------+------+--------+-----------+-------------+--------------+----+---+------+
|  0|  99|        2|      2|    99|      81|          3|           99|          5559|5742|100| 49300|
+---+----+---------+-------+------+--------+-----------+-------------+--------------+----+---+------+



In [27]:
users_spark.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in users_spark.columns
]).show()

+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+
| id|current_age|retirement_age|birth_year|birth_month|gender|address|latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+
|  0|         63|            65|        63|         55|    61|     68|      50|       54|               64|           46|        63|          49|              62|
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+



## Imputación de datos

In [28]:
# Imputar la edad actual utilizando la mediana

imputer = Imputer(
    inputCols=["current_age"],
    outputCols=["current_age"]
).setStrategy("median")

users_spark = imputer.fit(users_spark).transform(users_spark)

In [29]:
# Imputar el género utilizando la moda

gender_mode = (
    users_spark
    .filter(col("gender").isNotNull())
    .groupBy("gender")
    .count()
    .orderBy(col("count").desc())
    .first()["gender"]
)

users_spark = users_spark.fillna({
    "gender": gender_mode
})

## Verificar los cambios

In [30]:
users_spark.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in users_spark.columns
]).show()

+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+
| id|current_age|retirement_age|birth_year|birth_month|gender|address|latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+
|  0|          0|            65|        63|         55|     0|     68|      50|       54|               64|           46|        63|          49|              62|
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+

